# Data Cleaning – SaaS Dataset (synthetic data)
Keep this notebook in the same folder as the 6 raw CSV files, then run all cells.
Cleaned files are saved in a folder called `cleaned`.

**Steps for every table:** remove duplicates → fix text labels → fix dates → fix wrong numbers → remove rows with IDs that do not exist.

## 1. Import libraries and load the files

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('cleaned', exist_ok=True)
last_day = pd.Timestamp('2025-12-31')      # last date in the data

# dates come as 2025-03-14 or as 14/03/2025 -> try the first format, then the second
def fix_date(column):
    first = pd.to_datetime(column, format='%Y-%m-%d', errors='coerce')
    second = pd.to_datetime(column, format='%d/%m/%Y', errors='coerce')
    return first.fillna(second)

customers = pd.read_csv('customers.csv')
plans = pd.read_csv('plans.csv')
subs = pd.read_csv('subscriptions.csv')
events = pd.read_csv('subscription_events.csv')
invoices = pd.read_csv('invoices.csv')
marketing = pd.read_csv('marketing_spend.csv')

print(customers.shape, plans.shape, subs.shape, events.shape, invoices.shape, marketing.shape)

(10000, 7) (8, 7) (14000, 12) (32849, 11) (390000, 15) (10000, 11)


## 2. Subscriptions
| Column | Problem | Fix |
|---|---|---|
| all | duplicate rows | drop duplicates |
| status, billing_cycle | `Active`, `cancelled`, `yearly` | lowercase and one standard name |
| dates | `DD/MM/YYYY` format, end date before start date | convert to dates, wrong end date → empty |
| discount_pct | values like 150 or -10 | set to 0 |
| contract_term_months | annual plan with 1 month | set to 12 |
| cancellation_reason | empty for canceled | `unknown` |
| customer_id, plan_id | ID does not exist | remove row |

In [2]:
subs = subs.drop_duplicates()

# text columns
subs['status'] = subs['status'].str.strip().str.lower().str.replace(' ', '_')
subs['status'] = subs['status'].replace({'cancelled': 'canceled'})
subs['billing_cycle'] = subs['billing_cycle'].str.strip().str.lower()
subs['billing_cycle'] = subs['billing_cycle'].replace({'month': 'monthly', 'yearly': 'annual', 'annually': 'annual'})

# dates
for col in ['start_date', 'end_date', 'trial_start_date', 'trial_end_date']:
    subs[col] = fix_date(subs[col])
subs.loc[subs['end_date'] < subs['start_date'], 'end_date'] = pd.NaT
subs.loc[subs['trial_end_date'] < subs['trial_start_date'], 'trial_end_date'] = pd.NaT

# numbers
subs.loc[(subs['discount_pct'] < 0) | (subs['discount_pct'] > 30), 'discount_pct'] = 0
subs.loc[(subs['billing_cycle'] == 'annual') & (subs['contract_term_months'] < 12), 'contract_term_months'] = 12

# missing reason
subs.loc[(subs['status'] == 'canceled') & subs['cancellation_reason'].isna(), 'cancellation_reason'] = 'unknown'

# remove rows with IDs that do not exist
subs = subs[subs['customer_id'].isin(customers['customer_id'])]
subs = subs[subs['plan_id'].isin(plans['plan_id'])]
print(subs.shape)
subs['status'].value_counts()

(13873, 12)


status
active      9398
canceled    3968
past_due     397
trialing     110
Name: count, dtype: int64

## 3. Subscription events
| Column | Problem | Fix |
|---|---|---|
| all | duplicate rows | drop duplicates |
| event_type | `Upgrade`, `DOWNGRADE`, `canceled` | lowercase, `canceled` → `cancellation` |
| subscription_id | ID does not exist | remove row |
| event_date | wrong format, before subscription start | convert, replace with subscription start date |
| new_mrr | negative or too big | new_mrr = old_mrr + mrr_change |
| mrr_change | not equal to new − old | calculate again |
| old_plan_id | empty on upgrade/downgrade | use the previous plan of the same subscription |
| event_source | empty | `unknown` |
| customer_id | different from subscription owner | take from subscriptions |

In [3]:
events = events.drop_duplicates()

events['event_type'] = events['event_type'].str.strip().str.lower().str.replace(' ', '_')
events['event_type'] = events['event_type'].replace({'canceled': 'cancellation'})
events = events[events['subscription_id'].isin(subs['subscription_id'])]

# dates: wrong or too early -> subscription start date
start_dates = events['subscription_id'].map(subs.set_index('subscription_id')['start_date'])
events['event_date'] = fix_date(events['event_date'])
wrong_date = events['event_date'].isna() | (events['event_date'] < start_dates)
events.loc[wrong_date, 'event_date'] = start_dates

# MRR: negative or more than 3.5 x plan price is wrong
price = plans.set_index('plan_id')['monthly_list_price']
max_mrr = events['new_plan_id'].map(price).fillna(0) * 3.5
wrong_mrr = (events['new_mrr'] < 0) | (events['new_mrr'] > max_mrr)
events.loc[wrong_mrr, 'new_mrr'] = events['old_mrr'] + events['mrr_change']
events['mrr_change'] = (events['new_mrr'] - events['old_mrr']).round(2)

# old plan = previous new plan of the same subscription
events = events.sort_values(['subscription_id', 'event_date', 'event_id'])
previous_plan = events.groupby('subscription_id')['new_plan_id'].shift()
missing = events['old_plan_id'].isna() & events['event_type'].isin(['upgrade', 'downgrade'])
events.loc[missing, 'old_plan_id'] = previous_plan

events['event_source'] = events['event_source'].fillna('unknown')
events['customer_id'] = events['subscription_id'].map(subs.set_index('subscription_id')['customer_id'])

# canceled subscriptions without end date -> date of the cancellation event
cancel_date = events[events['event_type'].isin(['cancellation', 'trial_expired'])].groupby('subscription_id')['event_date'].max()
subs['end_date'] = subs['end_date'].fillna(subs['subscription_id'].map(cancel_date))
subs.loc[subs['status'] != 'canceled', 'end_date'] = pd.NaT
print(events.shape)
events['event_type'].value_counts()

(32418, 11)


event_type
expansion           7086
upgrade             5923
trial_conversion    3234
cross_sell          2561
new_subscription    2303
free_signup         2163
trial_expired       2088
cancellation        1878
downgrade           1576
reactivation        1399
renewal             1192
free_to_paid         569
contraction          446
Name: count, dtype: int64

## 4. Customers
| Column | Problem | Fix |
|---|---|---|
| all | duplicate rows | drop duplicates |
| acquisition_channel | `paid_search`, `PAID SEARCH` | proper words with capital letters |
| country | `USA`, `U.K.`, `Deu`, empty | full country name, empty → `Unknown` |
| company_size, industry | empty | `Unknown` |
| signup_date | wrong format, date in the future | convert, wrong → first subscription start date |
| lead_source_campaign_id | campaign does not exist | `UNKNOWN` |

In [4]:
customers = customers.drop_duplicates()

customers['acquisition_channel'] = customers['acquisition_channel'].str.strip().str.lower().str.replace('_', ' ').str.title()

customers['country'] = customers['country'].str.strip().str.title()
customers['country'] = customers['country'].replace({
    'Usa': 'United States', 'Us': 'United States', 'U.S.': 'United States',
    'Uk': 'United Kingdom', 'U.K.': 'United Kingdom', 'Great Britain': 'United Kingdom',
    'De': 'Germany', 'Deutschland': 'Germany', 'Fr': 'France', 'In': 'India',
    'Can': 'Canada', 'Aus': 'Australia', 'Bra': 'Brazil', 'Net': 'Netherlands',
    'Sin': 'Singapore', 'Spa': 'Spain', 'Jap': 'Japan'})
customers['country'] = customers['country'].fillna('Unknown')
customers['company_size'] = customers['company_size'].fillna('Unknown')
customers['industry'] = customers['industry'].fillna('Unknown')

customers['signup_date'] = fix_date(customers['signup_date'])
customers.loc[customers['signup_date'] > last_day, 'signup_date'] = pd.NaT
first_start = subs.groupby('customer_id')['start_date'].min()
customers['signup_date'] = customers['signup_date'].fillna(customers['customer_id'].map(first_start))
customers = customers.dropna(subset=['signup_date'])

customers.loc[~customers['lead_source_campaign_id'].isin(marketing['campaign_id']), 'lead_source_campaign_id'] = 'UNKNOWN'
print(customers.shape)
customers['acquisition_channel'].value_counts()

(9939, 7)


acquisition_channel
Organic Search       2217
Paid Search          1592
Direct               1394
Referral             1354
Paid Social          1305
Partnerships          822
Events & Webinars     682
Email Marketing       573
Name: count, dtype: int64

## 5. Invoices
| Column | Problem | Fix |
|---|---|---|
| all | duplicate rows | drop duplicates |
| subscription_id | ID does not exist | remove row |
| payment_status | `PAID`, `Partially Paid`, `past due`, `open` | paid, partial, failed, unpaid, overdue, refunded |
| payment_method | `CC`, `ACH`, `Wire`, empty | credit_card, paypal, ach_debit, wire_transfer, unknown |
| dates | wrong format, due date before invoice date, period end before start | convert and fix |
| amount_due | negative or 10 times too big | use amount_paid (paid invoices), negative → positive |
| tax_amount | negative, too big, empty | amount_due × normal tax rate of the country |
| discount_amount | negative or bigger than amount | 0 |
| amount_paid | empty or bigger than amount_due on paid invoices | = amount_due |
| payment_date | empty on paid invoices | due date |
| days_overdue | does not match status | calculate again |

In [5]:
invoices = invoices.drop_duplicates()
invoices = invoices[invoices['subscription_id'].isin(subs['subscription_id'])]

# text columns
invoices['payment_status'] = invoices['payment_status'].str.strip().str.lower()
invoices['payment_status'] = invoices['payment_status'].replace({
    'paid in full': 'paid', 'partially paid': 'partial', 'part_paid': 'partial',
    'payment_failed': 'failed', 'past due': 'overdue', 'open': 'unpaid', 'refund': 'refunded'})
invoices['payment_method'] = invoices['payment_method'].str.strip().str.lower()
invoices['payment_method'] = invoices['payment_method'].replace({
    'cc': 'credit_card', 'credit card': 'credit_card', 'creditcard': 'credit_card', 'card': 'credit_card',
    'ach': 'ach_debit', 'bank_transfer': 'ach_debit', 'wire': 'wire_transfer', 'wire transfer': 'wire_transfer'})
invoices['payment_method'] = invoices['payment_method'].fillna('unknown')

# dates
for col in ['invoice_date', 'billing_period_start', 'billing_period_end', 'payment_date', 'due_date']:
    invoices[col] = fix_date(invoices[col])

days_to_pay = invoices['payment_method'].map({'credit_card': 0, 'paypal': 0, 'ach_debit': 15, 'wire_transfer': 30, 'unknown': 0})
wrong = invoices['due_date'].isna() | (invoices['due_date'] < invoices['invoice_date'])
invoices.loc[wrong, 'due_date'] = invoices['invoice_date'] + pd.to_timedelta(days_to_pay, unit='D')
wrong = invoices['billing_period_end'] < invoices['billing_period_start']
invoices.loc[wrong, 'billing_period_end'] = invoices['billing_period_start'] + pd.Timedelta(days=30)

# amounts
paid = invoices['payment_status'] == 'paid'
wrong_due = paid & ((invoices['amount_due'] <= 0) | (invoices['amount_due'] > 5 * invoices['amount_paid']))
invoices.loc[wrong_due, 'amount_due'] = invoices['amount_paid']
invoices['amount_due'] = invoices['amount_due'].abs()
invoices.loc[paid, 'amount_paid'] = invoices['amount_due']
invoices.loc[(invoices['discount_amount'] < 0) | (invoices['discount_amount'] > invoices['amount_due']), 'discount_amount'] = 0
invoices.loc[paid & invoices['payment_date'].isna(), 'payment_date'] = invoices['due_date']

# tax: normal tax rate = median of tax / amount_due for each country
customer_of_sub = subs.set_index('subscription_id')['customer_id']
invoices['country'] = invoices['subscription_id'].map(customer_of_sub).map(customers.set_index('customer_id')['country'])
tax_rate = invoices['tax_amount'] / invoices['amount_due']
good_rate = tax_rate.where((tax_rate > 0.01) & (tax_rate < 0.25))
country_rate = good_rate.groupby(invoices['country']).transform('median').fillna(good_rate.median())
wrong_tax = invoices['tax_amount'].isna() | (invoices['tax_amount'] < 0) | (tax_rate >= 0.25)
invoices.loc[wrong_tax, 'tax_amount'] = (invoices['amount_due'] * country_rate).round(2)
invoices = invoices.drop(columns='country')

# days overdue
days_late = (invoices['payment_date'] - invoices['due_date']).dt.days.clip(lower=0)
days_open = (last_day - invoices['due_date']).dt.days.clip(lower=0)
invoices['days_overdue'] = np.where(invoices['payment_status'] == 'paid', days_late,
                           np.where(invoices['payment_status'] == 'refunded', 0, days_open)).astype(int)
print(invoices.shape)
invoices['payment_status'].value_counts()

(387149, 15)


payment_status
paid        349702
failed       10549
unpaid        9070
partial       7617
overdue       5798
refunded      4413
Name: count, dtype: int64

## 6. Marketing spend
| Column | Problem | Fix |
|---|---|---|
| all | duplicate rows | drop duplicates |
| acquisition_channel | label variants | proper words with capital letters |
| campaign_name | upper/lower case, `_` and spaces | UPPER CASE with `-` |
| country | empty | country of the same campaign, else `Unknown` |
| period_end | before period_start | last day of the month |
| marketing_spend | negative or 20 times too big | positive; too big → median cost of the channel |
| attributed_paid_customers | more than new customers | same as new customers |

In [6]:
marketing = marketing.drop_duplicates()

marketing['acquisition_channel'] = marketing['acquisition_channel'].str.strip().str.lower().str.replace('_', ' ').str.title()
marketing['campaign_name'] = marketing['campaign_name'].str.strip().str.replace('_', '-').str.upper()
marketing['country'] = marketing['country'].fillna(marketing.groupby('campaign_id')['country'].transform('first')).fillna('Unknown')

marketing['period_start'] = fix_date(marketing['period_start'])
marketing['period_end'] = marketing['period_start'] + pd.offsets.MonthEnd(0)

marketing['marketing_spend'] = marketing['marketing_spend'].abs()
cost = marketing['marketing_spend'] / (marketing['attributed_new_customers'] + 1)
median_cost = cost.groupby(marketing['acquisition_channel']).transform('median')
too_big = cost > 6 * median_cost
marketing.loc[too_big, 'marketing_spend'] = (median_cost * (marketing['attributed_new_customers'] + 1)).round(2)

wrong = marketing['attributed_paid_customers'] > marketing['attributed_new_customers']
marketing.loc[wrong, 'attributed_paid_customers'] = marketing['attributed_new_customers']
print(marketing.shape)
marketing.head()

(9900, 11)


,marketing_record_id,period_start,period_end,acquisition_channel,campaign_id,campaign_name,country,marketing_spend,attributed_new_customers,attributed_paid_customers,campaign_type
0,MKT-000001,2023-01-01,2023-01-31,Email Marketing,CMP-00070,EML-US-SMB-2023Q1,United States,280.87,3,2,Newsletter
1,MKT-000002,2023-01-01,2023-01-31,Paid Search,CMP-00102,PSR-SG-AUTOMATION-2023Q1,Singapore,230.84,0,0,Brand Search
2,MKT-000003,2023-01-01,2023-01-31,Organic Search,CMP-00125,ORG-US-SECURITY-2023Q1,United States,572.72,10,6,Organic Social
3,MKT-000004,2023-01-01,2023-01-31,Organic Search,CMP-00207,ORG-SG-ENTERPRISE-2023Q1,Singapore,363.62,8,7,SEO Content
4,MKT-000005,2023-01-01,2023-01-31,Paid Search,CMP-00259,PSR-DE-CORE-2023Q1,Germany,448.73,1,1,Brand Search


## 7. Save the cleaned files

In [7]:
plans.to_csv('cleaned/plans.csv', index=False)
customers.to_csv('cleaned/customers.csv', index=False)
subs.to_csv('cleaned/subscriptions.csv', index=False)
events.to_csv('cleaned/subscription_events.csv', index=False)
invoices.to_csv('cleaned/invoices.csv', index=False)
marketing.to_csv('cleaned/marketing_spend.csv', index=False)

# quick check: no ID without a parent
print('subscriptions without customer:', (~subs['customer_id'].isin(customers['customer_id'])).sum())
print('events without subscription   :', (~events['subscription_id'].isin(subs['subscription_id'])).sum())
print('invoices without subscription :', (~invoices['subscription_id'].isin(subs['subscription_id'])).sum())

subscriptions without customer: 0
events without subscription   : 0
invoices without subscription : 0
